# CSEC303 Workbench Dry Run

**Role:** Digital image laboratory trainee  
**Goal:** Practice the same Custos workflow used in the laboratory midterm without using the real PHANTOM-303 answers.

Run the setup cell, move through Cases 00-03, edit the marked values, and answer the matching Custos items. You may rerun cells as often as needed.


## How to work
- Use **Run Cell** to execute Python directly in Custos.
- Change only values marked `TRY ME` unless you want to experiment further.
- Your code edits are saved with your Custos attempt.
- Use the plots and printed metrics as evidence for your answers.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=5, suppress=True)

def mse(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    return float(np.mean((a-b)**2))

print("Workbench ready")


## CASE 00 · Pixel Evidence
An 8-bit grayscale image stores one unsigned intensity value per pixel. Use the cell to inspect the number of representable values and a simple gradient.


In [ ]:
levels = 2 ** 8
img8 = np.arange(256, dtype=np.uint8).reshape(16, 16)
print("Intensity levels:", levels)
print("Minimum:", img8.min(), "Maximum:", img8.max())
plt.imshow(img8, cmap="gray", vmin=0, vmax=255)
plt.title("8-bit intensity gradient")
plt.axis("off")
plt.show()


## CASE 01 · Contrast Recovery
A reference gradient was darkened with a power-law transformation. Try the candidate gamma values and identify which value most closely restores the reference.


In [ ]:
reference = np.linspace(0, 1, 64*64).reshape(64, 64)
dark = reference ** 2.2
candidates = [0.35, 0.45, 0.60, 1.00]

for gamma in candidates:
    restored = dark ** gamma
    print(f"gamma={gamma:0.2f}  MSE={mse(restored, reference):.7f}")

selected_gamma = 0.45  # TRY ME
restored = dark ** selected_gamma
fig, ax = plt.subplots(1, 3, figsize=(9, 3))
for a, im, title in zip(ax, [reference, dark, restored], ["Reference", "Darkened", f"gamma={selected_gamma}"]):
    a.imshow(im, cmap="gray", vmin=0, vmax=1); a.set_title(title); a.axis("off")
plt.tight_layout(); plt.show()


## CASE 02 · Noise Cleanup
The synthetic evidence below contains salt-and-pepper noise. Test odd median-kernel sizes. The best candidate for this exercise is the one with the lowest MSE against the known clean reference.


In [ ]:
rng = np.random.default_rng(303)
clean = np.full((64, 64), 0.20)
clean[16:48, 16:48] = 0.80
noisy = clean.copy()
mask = rng.random(clean.shape)
noisy[mask < 0.025] = 0.0
noisy[(mask >= 0.025) & (mask < 0.05)] = 1.0

def median_filter_small(img, k):
    p = k // 2
    padded = np.pad(img, p, mode="edge")
    out = np.empty_like(img)
    for y in range(img.shape[0]):
        for x in range(img.shape[1]):
            out[y, x] = np.median(padded[y:y+k, x:x+k])
    return out

for k in [1, 3, 5, 7]:
    candidate = median_filter_small(noisy, k)
    print(f"kernel={k}x{k}  MSE={mse(candidate, clean):.7f}")

selected_kernel = 3  # TRY ME
filtered = median_filter_small(noisy, selected_kernel)
fig, ax = plt.subplots(1, 3, figsize=(9, 3))
for a, im, title in zip(ax, [clean, noisy, filtered], ["Clean", "Noisy", f"Median {selected_kernel}x{selected_kernel}"]):
    a.imshow(im, cmap="gray", vmin=0, vmax=1); a.set_title(title); a.axis("off")
plt.tight_layout(); plt.show()


## CASE 03 · Frequency Clue
A periodic vertical interference pattern was added to a smooth image. Inspect the shifted FFT magnitude and locate the symmetric interference peaks relative to the spectrum center.


In [ ]:
n = 128
x = np.arange(n)
base = np.full((n, n), 0.45)
periodic = 0.20 * np.sin(2*np.pi*8*x/n)
noisy_frequency = np.clip(base + periodic[None, :], 0, 1)
F = np.fft.fftshift(np.fft.fft2(noisy_frequency - noisy_frequency.mean()))
mag = np.log1p(np.abs(F))
cy, cx = np.array(mag.shape) // 2
mask = np.ones_like(mag, dtype=bool)
mask[cy-2:cy+3, cx-2:cx+3] = False
coords = np.argwhere(mask)
values = mag[mask]
order = np.argsort(values)[-8:][::-1]
print("Strongest offsets from center (dy, dx):")
seen=[]
for y, x0 in coords[order]:
    off=(int(y-cy), int(x0-cx))
    if off not in seen:
        seen.append(off); print(off)
plt.imshow(mag, cmap="magma")
plt.title("Shifted log-magnitude spectrum")
plt.axis("off")
plt.show()


## Final Check
You should now have enough evidence to answer all six dry-run items in Custos. This practice assessment may reveal its answer review immediately after submission.
